# Character Error Rate (CER) Calculation
Calculate CER accuracy for Transkribus OCR model using METS.xml structure

In [12]:
import json
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd

In [13]:
# Paths to your data
corpus_path = Path("/Users/sophiehamann/Documents/MA_Universität_Wien/master_thesis/master_thesis_data/processed/full_corpus.json")
ground_truth_dir = Path("/Users/sophiehamann/Documents/MA_Universität_Wien/master_thesis/master_thesis_data/processed/export_job_22502221/11906961/heresies_ground_truth")
page_dir = ground_truth_dir / "page"

# Load corpus
with open(corpus_path, 'r', encoding='utf-8') as f:
    corpus = json.load(f)

print(f"Corpus loaded: {len(corpus)} items")
print(f"Page directory: {page_dir}")
xml_files = sorted(list(page_dir.glob('*.xml')))
print(f"XML files found: {len(xml_files)}")

Corpus loaded: 33 items
Page directory: /Users/sophiehamann/Documents/MA_Universität_Wien/master_thesis/master_thesis_data/processed/export_job_22502221/11906961/heresies_ground_truth/page
XML files found: 135


In [14]:
# DEBUG: Check the actual XML structure
if xml_files:
    sample_file = xml_files[0]
    print(f"Examining: {sample_file.name}\n")
    
    tree = ET.parse(sample_file)
    root = tree.getroot()
    
    # Print root tag and namespace
    print(f"Root tag: {root.tag}")
    print(f"Root attrib: {root.attrib}\n")
    
    # Extract namespace from root tag if present
    if '}' in root.tag:
        namespace = root.tag.split('}')[0] + '}'
        print(f"Detected namespace: {namespace}")
    
    # Print first few children
    print("\nFirst-level children:")
    for i, child in enumerate(root):
        print(f"  {i}: {child.tag}")
        if i > 5:
            print("  ...")
            break
    
    # Find all Unicode elements (ignore namespace for now)
    print("\nSearching for all Unicode elements...")
    all_unicode = [elem for elem in root.iter() if 'Unicode' in elem.tag]
    print(f"Found {len(all_unicode)} Unicode elements")
    
    if all_unicode:
        print(f"\nFirst 5 Unicode elements:")
        for elem in all_unicode[:5]:
            print(f"  Text: '{elem.text}'")
            print(f"  Parent: {elem.find('..').tag if hasattr(elem, 'find') else 'N/A'}")

Examining: 0001_p002.xml

Root tag: {http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}PcGts
Root attrib: {'{http://www.w3.org/2001/XMLSchema-instance}schemaLocation': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15 http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15/pagecontent.xsd'}

Detected namespace: {http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}

First-level children:
  0: {http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}Metadata
  1: {http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}Page

Searching for all Unicode elements...
Found 118 Unicode elements

First 5 Unicode elements:
  Text: '30'


AttributeError: 'NoneType' object has no attribute 'tag'

In [15]:
# Simple extraction: find all Unicode elements regardless of namespace
def extract_text_from_xml_simple(xml_path):
    """Extract text from Transkribus PAGE XML - simple version without namespace"""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        # Find all elements with 'Unicode' in the tag name
        text_lines = []
        for elem in root.iter():
            if 'Unicode' in elem.tag:
                if elem.text and elem.text.strip():
                    text_lines.append(elem.text)
        
        return ' '.join(text_lines)
    except Exception as e:
        print(f"Error reading {xml_path}: {e}")
        return ""

# Test it
if xml_files:
    sample_text = extract_text_from_xml_simple(xml_files[0])
    print(f"\nExtracted text from {xml_files[0].name}:")
    print(sample_text[:500] if sample_text else "Still no text found!")
    print(f"\nTotal characters: {len(sample_text)}")


Extracted text from 0001_p002.xml:
30 The Art of Not Bowing: Writing by Women in Prison Carol Muske Who the hell am I anyway Not to bow? (Assata Shakur/Joanne Chesimard) In July 1973 I wrote an article for The Village Voice about a hunger strike then taking place at the Women's House of Detention (New York City Correctional Institution for Women, hous- ing around 400 detention and sentenced wom- en) on Riker´s Island. I used a pseudonym for the article because I was working at the time at the prison as a mental health worker as w

Total characters: 4350


In [16]:
# Levenshtein distance function
def levenshtein_distance(s1, s2):
    """Calculate Levenshtein distance between two strings"""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    
    if len(s2) == 0:
        return len(s1)
    
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]

def calculate_cer(predicted, reference):
    """Calculate Character Error Rate"""
    distance = levenshtein_distance(predicted, reference)
    cer = distance / len(reference) if len(reference) > 0 else 0
    return cer, distance

In [17]:
# Calculate CER for all pages using simple extraction
results = []

for xml_file in sorted(page_dir.glob('*.xml')):
    page_num = xml_file.stem  # filename without extension
    
    # Get ground truth from XML
    ground_truth = extract_text_from_xml_simple(xml_file)
    
    # Get predicted text from corpus
    predicted = ""
    for item in corpus:
        item_id = str(item.get('id', '') or item.get('page_id', '') or item.get('page', ''))
        if page_num in item_id:
            predicted = item.get('text', '') or item.get('content', '')
            break
    
    if ground_truth:
        if predicted:
            cer, distance = calculate_cer(predicted, ground_truth)
            results.append({
                'page': page_num,
                'ground_truth_length': len(ground_truth),
                'predicted_length': len(predicted),
                'levenshtein_distance': distance,
                'CER': cer
            })
            print(f"Page {page_num}: CER = {cer:.4f}")
        else:
            print(f"Page {page_num}: No matching text in corpus")
    else:
        print(f"Page {page_num}: No ground truth text")

# Create results dataframe
df_results = pd.DataFrame(results)
print("\n" + "="*60)
print(df_results)

Page 0001_p002: No matching text in corpus
Page 0002_p003: No matching text in corpus
Page 0003_p004: No matching text in corpus
Page 0004_p005: No matching text in corpus
Page 0005_p006: No matching text in corpus
Page 0006_p007: No matching text in corpus
Page 0007_p008: No matching text in corpus
Page 0008_p009: No matching text in corpus
Page 0009_p010: No matching text in corpus
Page 0010_p011: No matching text in corpus
Page 0011_p012: No matching text in corpus
Page 0012_p013: No matching text in corpus
Page 0013_p014: No matching text in corpus
Page 0014_p015: No matching text in corpus
Page 0015_p016: No matching text in corpus
Page 0016_p017: No matching text in corpus
Page 0017_p018: No matching text in corpus
Page 0018_p019: No matching text in corpus
Page 0019_p020: No matching text in corpus
Page 0020_p021: No matching text in corpus
Page 0021_p022: No matching text in corpus
Page 0022_p023: No matching text in corpus
Page 0023_p024: No matching text in corpus
Page 0024_p

In [ ]:
# Summary statistics
if len(df_results) > 0:
    print("\n" + "="*60)
    print("OVERALL CER STATISTICS")
    print("="*60)
    print(f"Total pages: {len(df_results)}")
    print(f"Mean CER: {df_results['CER'].mean():.4f}")
    print(f"Median CER: {df_results['CER'].median():.4f}")
    print(f"Min CER: {df_results['CER'].min():.4f}")
    print(f"Max CER: {df_results['CER'].max():.4f}")
    print(f"Std Dev: {df_results['CER'].std():.4f}")
    print(f"\nAccuracy (1 - CER): {(1 - df_results['CER'].mean())*100:.2f}%")
else:
    print("No results to display")

In [ ]:
# Optional: Save results to CSV
output_path = Path("cer_results.csv")
df_results.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")